# 9장 실습 — 통행시간 예측 모델

배차는 "어느 차가 가장 빨리 오는가"를 알아야 합니다.
그 값을 매번 다익스트라로 구하면 시뮬레이션 한 번에 라우팅이 3만 번 넘게 필요합니다.
이 실습에서는 좌표와 시각만으로 소요시간을 예측하는 모델을 만듭니다. 교재 9장에 대응합니다.

순서는 교재와 같습니다. 정답 데이터 → 기준선 → 선형회귀 → LightGBM → 오차 분석 → 배차 비용행렬.

`scikit-learn` 과 `lightgbm` 이 필요합니다.

```bash
pip install -r requirements-heavy.txt
```

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 정답을 만듭니다 (교재 9.1)

지도학습에는 정답이 필요합니다. 정답은 3장의 다익스트라가 만듭니다.
`build_dataset` 은 도로망 노드 둘을 무작위로 뽑고, 최단경로 소요시간을 구해 한 행으로 적습니다.

2만 건이면 3분쯤 걸리므로 교재가 만들어 둔 `data/hanam/eta_samples.parquet` 을 읽습니다.
직접 만들려면 `build_dataset("hanam", n=20_000, seed=0)` 을 부릅니다.

In [ ]:
import pandas as pd

from smartmob.data import data_path
from smartmob.teaching.eta import FEATURES, TARGET

df = pd.read_parquet(data_path("hanam/eta_samples.parquet"))
print(f"{len(df):,}건")
print("특징:", FEATURES)
print("정답:", TARGET)
df.head(3).round(3)

`duration_min` 이 정답입니다. `network_km`(도로 거리)은 라우팅을 해야 나오므로 특징에 넣지 않습니다.
예측할 때 얻을 수 없는 값을 학습에 넣는 것을 데이터 누수(data leakage)라고 합니다.
학습할 때는 잘 맞다가 실제로 쓸 때 그 값이 없어서 무너집니다.

특징 여덟 개는 전부 좌표와 시각만으로 계산합니다.

| 특징 | 왜 넣는가 |
|---|---|
| `straight_km` | 거리가 늘면 시간이 늡니다 |
| `hour` | 4장에서 본 시간대별 속도 차이 |
| `sin_bearing`, `cos_bearing` | 방향. 한강을 건너는 남북과 강변을 따르는 동서가 다릅니다 |
| `origin_lat/lon`, `dest_lat/lon` | 어느 지역인지. 시가지와 외곽의 도로 사정이 다릅니다 |

방위각을 `sin`, `cos` 둘로 나눈 이유는 각도가 원 위의 값이기 때문입니다.
359도와 1도는 거의 같은 방향인데 숫자로는 358만큼 떨어져 있습니다.

데이터를 학습용과 검증용으로 나눕니다. 학습에 쓴 데이터로 평가하면,
그 데이터에서만 잘 맞는 것인지 새 데이터에서도 맞는 것인지 알 수 없습니다.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURES]      # 입력(특징) 8개 열
y = df[TARGET]        # 정답 1개 열

# test_size=0.2: 20% 를 검증용으로 떼어 둡니다. random_state 를 고정하면 매번 같은 행이 뽑힙니다.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"학습 {len(X_train):,}건, 검증 {len(X_test):,}건")

## 2. 기준선 (교재 9.2)

모델을 만들기 전에 기준선을 정합니다. 기준선보다 못한 모델은 쓸 이유가 없습니다.

가장 단순한 예측은 직선거리 ÷ 평균 속도입니다.
평균 속도는 학습 데이터의 직선거리 합을 소요시간 합으로 나눈 값입니다.
통행마다 속도를 구해 평균 내는 것이 아니라 전체 거리를 전체 시간으로 나눕니다.

평가 지표는 둘입니다.

- MAE(평균절대오차) — 평균 몇 분 틀리는가. 단위가 분이라 바로 읽힙니다
- R²(결정계수) — 정답의 변동 중 몇 %를 설명하는가. 1이 최선이고 0이면 평균값을 답하는 것과 같습니다

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

avg_speed = X_train["straight_km"].sum() / (y_train.sum() / 60)     # km ÷ 시간 = km/h
baseline = X_test["straight_km"] / avg_speed * 60                   # km ÷ (km/h) × 60 = 분

scores = {}                                                          # 모델별 MAE 를 모아 둡니다
scores["기준선"] = mean_absolute_error(y_test, baseline)

print(f"평균 속도 {avg_speed:.1f} km/h")
print(f"MAE {scores['기준선']:.2f}분   R² {r2_score(y_test, baseline):.3f}")

평균 27.3km/h 로 나누면 2분 31초 틀립니다. 통행 평균이 11.8분이므로 20% 넘게 틀리는 셈입니다.

## 3. 선형회귀 (교재 9.3)

직선거리 하나로 직선을 맞춘 뒤, 특징 여덟 개를 다 넣어 봅니다.
`fit` 이 학습이고 `predict` 가 예측입니다. `scikit-learn` 의 모든 모델이 이 두 이름을 씁니다.

In [ ]:
from sklearn.linear_model import LinearRegression

lr1 = LinearRegression().fit(X_train[["straight_km"]], y_train)      # 특징 1개
pred1 = lr1.predict(X_test[["straight_km"]])
print(f"직선거리만   기울기 {lr1.coef_[0]:.2f}분/km, 절편 {lr1.intercept_:.2f}분")
print(f"             MAE {mean_absolute_error(y_test, pred1):.2f}분   R² {r2_score(y_test, pred1):.3f}")

lr2 = LinearRegression().fit(X_train, y_train)                       # 특징 8개 전부
pred2 = lr2.predict(X_test)
scores["선형회귀"] = mean_absolute_error(y_test, pred2)
print(f"특징 8개     MAE {scores['선형회귀']:.2f}분   R² {r2_score(y_test, pred2):.3f}")

기준선보다 조금 나아진 이유는 절편입니다. 거리가 0이어도 출발과 도착에 시간이 듭니다.

특징을 일곱 개 더 줘도 2.19분에서 2.16분으로 거의 그대로입니다.
선형회귀는 각 특징이 결과에 일정한 비율로 기여한다고 가정합니다.
실제로는 시가지의 위도 0.01과 외곽의 위도 0.01이 다릅니다. 위치는 다른 특징과 얽혀서 작동합니다.

## 4. 그래디언트 부스팅 (교재 9.4)

얽힌 관계를 다루려면 모델을 바꿔야 합니다.
의사결정나무는 데이터를 조건으로 쪼개고 조각마다 평균을 답합니다.
가장 작은 나무는 조건 하나짜리입니다. 직선거리 하나로 만들어 봅니다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# max_depth=1: 질문을 한 번만 하는 나무. 데이터를 두 조각으로 나눕니다.
stump = DecisionTreeRegressor(max_depth=1).fit(X_train[["straight_km"]], y_train)
split = stump.tree_.threshold[0]                   # 어디서 쪼갰는가
short, long = stump.tree_.value[1][0][0], stump.tree_.value[2][0][0]   # 두 조각의 평균
print(f"직선거리가 {split:.2f}km 보다 짧으면 {short:.1f}분, 길면 {long:.1f}분")
print(f"MAE {mean_absolute_error(y_test, stump.predict(X_test[['straight_km']])):.2f}분")

5.55km 를 기준으로 8.0분 아니면 16.6분을 답합니다. 질문 하나로는 3분 넘게 틀립니다.

나무 하나는 약하지만, 앞 나무가 틀린 만큼을 다음 나무가 맞추도록 수백 개를 이어 붙이면 강해집니다.
이것이 그래디언트 부스팅이고, LightGBM 은 그 구현 중 하나입니다. 인자는 셋만 알면 됩니다.

- `n_estimators` — 나무 개수
- `learning_rate` — 나무 하나가 앞의 오차를 얼마나 고치는가. 작을수록 나무가 더 필요합니다
- `num_leaves` — 나무 하나가 데이터를 몇 조각으로 나누는가

In [ ]:
import time

import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=400, learning_rate=0.05, num_leaves=31,
    random_state=42, verbose=-1,          # verbose=-1: 학습 로그를 끕니다
)

t0 = time.perf_counter()
model.fit(X_train, y_train)
train_time = time.perf_counter() - t0

pred3 = model.predict(X_test)
scores["LightGBM"] = mean_absolute_error(y_test, pred3)
print(f"MAE {scores['LightGBM']:.2f}분   R² {r2_score(y_test, pred3):.3f}   학습 {train_time:.1f}초")

board = pd.DataFrame({"MAE(분)": scores}).round(2)
board["기준선 대비"] = (board["MAE(분)"] / scores["기준선"]).round(2)
board

MAE 0.89분, R² 0.954 입니다. 기준선 2.52분 → 선형회귀 2.16분 → LightGBM 0.89분으로, 평균 1분 이내로 맞힙니다.

## 5. 무엇이 예측에 쓰였는가 (교재 9.5)

`feature_importances_` 는 그 특징으로 데이터를 쪼갠 횟수입니다.
예측을 얼마나 좋게 했는가가 아니라 몇 번 쓰였는가입니다.

In [ ]:
import matplotlib.pyplot as plt

split_count = pd.Series(model.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(split_count.index, split_count.values, color="tab:blue")
ax.set_xlabel("분기 횟수")
ax.set_title("LightGBM 이 각 특징으로 데이터를 쪼갠 횟수")
fig.tight_layout();

좌표 네 개가 직선거리보다 많이 쓰였습니다.
좌표만 있으면 거리도 계산되고 지역별 도로 사정까지 함께 배울 수 있습니다.
값이 촘촘한 특징은 잘게 여러 번 쪼개기 좋아서 횟수로 세면 크게 나옵니다. 이 순서가 곧 중요도는 아닙니다.

`hour` 가 가장 적게 쓰였습니다. 시간대별 평균 소요시간이 11.0분에서 12.7분 사이로 차이가 1.7분뿐이기 때문입니다.
4장의 62% 차이는 특정 구간의 이야기였고, 도시 전체 평균으로 보면 훨씬 작습니다.

## 6. 얼마나 빨라졌는가 (교재 9.6)

모델을 만든 이유는 정확도가 아니라 속도였습니다. 한 건 예측에 걸리는 시간을 다익스트라와 비교합니다.

In [ ]:
sample = X_test.head(2000)

t0 = time.perf_counter()
model.predict(sample)
per_query_us = (time.perf_counter() - t0) / len(sample) * 1e6     # 1e6: 초 → 마이크로초

print(f"모델 예측     {per_query_us:8.1f} µs/건")
print(f"다익스트라    {7400:8.1f} µs/건  (3.6절에서 잰 값)")
print(f"                 {7400 / per_query_us:,.0f}배 빠릅니다")

3천 배 안팎입니다. 4장에서 걱정한 "시뮬레이션 한 번에 몇 분"이 몇 초가 됩니다. 대신 1분쯤 틀립니다.

- 배차 결정: 받아들입니다. 누가 가장 가까운지만 알면 되고, 1분 오차로 순위가 뒤집히는 경우는 드뭅니다
- 승객에게 보여 주는 도착 예정시간: 곤란합니다. 실제 경로를 계산해야 합니다
- 차량이 실제로 움직이는 경로: 안 됩니다. 좌표열이 있어야 지도에 그립니다

## 7. 어디서 틀리는가 (교재 9.7)

평균 오차 한 줄로는 모델을 믿을 수 없습니다. 어떤 통행에서 크게 틀리는지 봅니다.

In [ ]:
error = pred3 - y_test           # 양수면 실제보다 길게, 음수면 짧게 예측한 것입니다

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(y_test, pred3, s=3, alpha=0.2, color="tab:blue")
lim = [0, y_test.max()]
axes[0].plot(lim, lim, color="tab:red", linewidth=1.2, linestyle="--")   # 대각선 = 완벽한 예측
axes[0].set_xlabel("실제 (분)"); axes[0].set_ylabel("예측 (분)")
axes[0].set_title("예측 대 실제")

axes[1].scatter(X_test["straight_km"], error, s=3, alpha=0.2, color="tab:blue")
axes[1].axhline(0, color="tab:red", linewidth=1.2, linestyle="--")
axes[1].set_xlabel("직선거리 (km)"); axes[1].set_ylabel("오차 (분)")
axes[1].set_title("거리별 오차")

for ax in axes:
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout();

왼쪽 그림에서 점들이 대각선에 붙어 있고, 오래 걸리는 통행일수록 흩어집니다.
가장 크게 틀린 다섯 건을 도로 거리와 함께 봅니다.

In [ ]:
# assign: 검증 표에 열 몇 개를 덧붙인 새 표를 만듭니다. network_km 은 원본 df 에서 같은 행을 가져옵니다.
check = X_test[["straight_km", "hour"]].assign(
    network_km=df.loc[X_test.index, "network_km"],
    실제=y_test, 예측=pred3.round(1), 오차=abs(error).round(1),
)
check.nlargest(5, "오차")        # 오차 열이 큰 순서로 5행

`network_km` 이 `straight_km` 보다 훨씬 큰 통행들입니다. 직선으로 1.4km 인데 도로로 19km 를 돌아가는 경우도 있습니다.
강 건너편이거나 산을 우회하는 곳입니다. 좌표만 보는 모델이 이것을 알아내기 어렵습니다.

## 8. 배차 비용행렬에 넣어 보기 (교재 10.6 미리 보기)

이 모델이 실제로 쓰이는 자리는 10장의 배차 비용행렬입니다.
승객 6명 × 차량 8대 행렬을 직선거리, 모델, 실제 라우팅 세 가지로 만들어 시간과 오차를 잽니다.
무작위 점이 도로로 닿지 않으면 라우팅 행렬에 무한대가 생기므로, 그런 쌍이 없을 때까지 다시 뽑습니다.

In [ ]:
import random

import numpy as np

from smartmob.data import load_road_graph
from smartmob.teaching.dispatch import cost_matrix, cost_matrix_from_model, cost_matrix_from_router

G = load_road_graph("hanam", modes=("drive",))
rng = random.Random(3)                     # 교재 10.6절과 같은 표본이 나오도록 씨앗을 맞춥니다

def random_points(n):
    """하남 일대를 감싸는 사각형 안의 무작위 점 n개."""
    return [(37.50 + rng.random() * 0.10, 127.13 + rng.random() * 0.14) for _ in range(n)]

while True:
    P, V = random_points(6), random_points(8)
    t0 = time.perf_counter()
    c_router = cost_matrix_from_router(P, V, G)
    t_router = time.perf_counter() - t0
    if np.isfinite(c_router).all():        # 도로로 못 닿는 쌍이 없을 때까지
        break

t0 = time.perf_counter(); c_straight = cost_matrix(P, V);                       t_straight = time.perf_counter() - t0
t0 = time.perf_counter(); c_model = cost_matrix_from_model(P, V, model.predict); t_model = time.perf_counter() - t0

banner("6×8 비용행렬 48칸: 걸린 시간과 라우팅 대비 칸당 오차")
print(f"직선거리    {t_straight * 1000:7.2f} ms   오차 {np.abs(c_straight - c_router).mean():.2f}분")
print(f"ETA 모델    {t_model * 1000:7.2f} ms   오차 {np.abs(c_model - c_router).mean():.2f}분")
print(f"실제 라우팅 {t_router * 1000:7.2f} ms")

라우팅은 48칸에 수백 ms 가 걸리고, 모델은 몇 ms 입니다.
칸당 오차는 모델이 직선거리보다 작지만 둘 다 4분 안팎입니다. 9.7절에서 본 외곽 통행이 섞여 있기 때문입니다.
배차에서 중요한 것은 칸의 값이 아니라 어느 차를 고르는가입니다. 그 비교는 10장 실습 7절에서 합니다.

## 9. 빈칸

### 9.1 특징 하나 추가하기

`FEATURES` 에 없는 특징을 하나 만들어 넣고 MAE 가 줄어드는지 봅니다.
후보는 출발지와 목적지의 위도 차이, 경도 차이, 시가지 중심까지의 거리 같은 것입니다.
라우팅 결과는 쓰지 않습니다. 그러면 모델을 쓰는 의미가 없어집니다.

In [ ]:
my_feature_name = None      # 추가한 특징의 이름
my_feature_mae = None       # 그때의 MAE (분)

banner("빈칸 9.1")
todo("추가한 특징", my_feature_name)
todo("그때의 MAE", my_feature_mae, fmt=lambda v: f"{v:.2f}분")

### 9.2 특징 하나 빼기

`hour` 를 빼고 학습하면 MAE 가 얼마나 나빠지는지, `straight_km` 을 빼면 얼마나 나빠지는지 잽니다.
특징을 하나씩 빼 가며 MAE 변화를 재는 것을 제거 실험(ablation)이라고 합니다.
5절의 분기 횟수 순서와 결과가 일치하는지 두 줄로 적습니다.

In [ ]:
mae_without_hour = None         # hour 를 뺐을 때의 MAE (분)
mae_without_straight = None     # straight_km 을 뺐을 때의 MAE (분)

banner("빈칸 9.2")
todo("hour 없이", mae_without_hour, fmt=lambda v: f"{v:.2f}분")
todo("straight_km 없이", mae_without_straight, fmt=lambda v: f"{v:.2f}분")

### 9.3 가장 크게 틀린 통행의 위치

7절의 다섯 건을 지도에 찍어 출발지와 목적지가 어디인지 봅니다.
왜 그 통행에서 크게 틀렸는지 한 줄로 적습니다. 한강과 검단산의 위치를 함께 봅니다.

In [ ]:
worst_error_min = None      # 가장 큰 절대오차 (분)

banner("빈칸 9.3")
todo("가장 큰 오차", worst_error_min, fmt=lambda v: f"{v:.1f}분")

## 정리

- 정답은 3장의 최단경로로 만듭니다. 특징은 예측 시점에 얻을 수 있는 것만 씁니다
- 기준선(직선거리 ÷ 27.3km/h) MAE 2.52분 → 선형회귀 2.16분 → LightGBM 0.89분
- 선형회귀는 특징을 더 줘도 나아지지 않습니다. 위치는 다른 특징과 얽혀 작동합니다
- 분기 횟수는 몇 번 쓰였는가이지 얼마나 중요한가가 아닙니다
- 모델은 다익스트라보다 3천 배 빠르고 1분쯤 틀립니다. 오차는 우회가 큰 통행에서 큽니다
- 10장 실습에서 이 예측값으로 만든 비용행렬이 라우팅과 같은 차를 고르는지 봅니다